In [2]:
import polars as pl
import numpy as np

clinical = pl.read_csv(f"../dataset/original/clinical.csv")
clinical = clinical.filter(pl.col('GEX_data') == 'yes')
clinical = clinical.drop(['id', 'creation_datetime', 'original_patientID', 'OS_status', 'OS_month', 'CNA_data', 'SNV_data', 'GEX_data'])
clinical

patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_status,PFS_month,drug,BOR,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source
str,str,i64,str,str,str,str,f64,str,str,str,str,str,str,str
"""YR_3053""","""male""",26,"""IV""","""M1B""","""elevated""","""1""",2.9,"""vemurafenib""","""PD""","""V600E""",null,"""no""","""no""","""doi:10.1158/1078-0432.CCR-18-0…"
"""YR_3151""","""female""",28,"""IV""","""M1C""","""normal""","""1""",1.9,"""vemurafenib""","""PD""","""V600E""",null,"""no""","""no""","""doi:10.1158/1078-0432.CCR-18-0…"
"""YR_3708""","""female""",58,"""IV""","""M1A""","""normal""","""0""",13.2,"""vemurafenib""","""CR""","""V600E""",null,"""no""","""no""","""doi:10.1158/1078-0432.CCR-18-0…"
"""YR_3705""","""male""",45,"""IV""","""M1A""","""normal""","""0""",17.4,"""vemurafenib""","""CR""","""V600E""",null,"""no""","""no""","""doi:10.1158/1078-0432.CCR-18-0…"
"""YR_1006""","""male""",53,"""IV""","""M1C""","""elevated""","""1""",1.4,"""vemurafenib + cobimetinib""","""PD""","""V600E""",null,"""no""","""no""","""doi:10.1158/1078-0432.CCR-18-0…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HL_40""","""male""",47,"""IV""","""M1C""",null,"""1""",3.0,"""vemurafenib""","""PR""","""V600E""","""no""","""no""","""no""","""doi:10.1016/j.cell.2015.07.061"""
"""HL_41""","""male""",39,"""IV""","""M1A""",null,"""1""",4.0,"""vemurafenib""","""PR""","""V600E""","""no""","""no""","""no""","""doi:10.1016/j.cell.2015.07.061"""
"""HL_42""","""male""",84,"""IV""","""M1C""",null,"""1""",8.0,"""dabrafenib""","""PR""","""V600E""","""no""","""no""","""no""","""doi:10.1016/j.cell.2015.07.061"""


In [3]:
source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.',
    "doi:10.3390/cancers12082224": 'Blateau et al.',
    "doi:10.1200/PO.16.00054": 'Catalanotti et al.',
    'doi:10.1158/2159-8290.CD-13-0617': 'Van Allen at al.'
}

clinical = clinical.with_columns(pl.col('source').replace(source_map))
clinical

patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_status,PFS_month,drug,BOR,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source
str,str,i64,str,str,str,str,f64,str,str,str,str,str,str,str
"""YR_3053""","""male""",26,"""IV""","""M1B""","""elevated""","""1""",2.9,"""vemurafenib""","""PD""","""V600E""",null,"""no""","""no""","""Yan et al."""
"""YR_3151""","""female""",28,"""IV""","""M1C""","""normal""","""1""",1.9,"""vemurafenib""","""PD""","""V600E""",null,"""no""","""no""","""Yan et al."""
"""YR_3708""","""female""",58,"""IV""","""M1A""","""normal""","""0""",13.2,"""vemurafenib""","""CR""","""V600E""",null,"""no""","""no""","""Yan et al."""
"""YR_3705""","""male""",45,"""IV""","""M1A""","""normal""","""0""",17.4,"""vemurafenib""","""CR""","""V600E""",null,"""no""","""no""","""Yan et al."""
"""YR_1006""","""male""",53,"""IV""","""M1C""","""elevated""","""1""",1.4,"""vemurafenib + cobimetinib""","""PD""","""V600E""",null,"""no""","""no""","""Yan et al."""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HL_40""","""male""",47,"""IV""","""M1C""",null,"""1""",3.0,"""vemurafenib""","""PR""","""V600E""","""no""","""no""","""no""","""Hugo et al."""
"""HL_41""","""male""",39,"""IV""","""M1A""",null,"""1""",4.0,"""vemurafenib""","""PR""","""V600E""","""no""","""no""","""no""","""Hugo et al."""
"""HL_42""","""male""",84,"""IV""","""M1C""",null,"""1""",8.0,"""dabrafenib""","""PR""","""V600E""","""no""","""no""","""no""","""Hugo et al."""


In [12]:
clinical.select(pl.col('source').unique())

source
str
"""Hugo et al."""
"""Long et al."""
"""Yan et al."""
"""Kwong et al."""
"""Catalanotti et al."""
"""Louveau et al."""
"""Blateau et al."""
"""Van Allen at al."""
"""Rizos et al."""


In [13]:
# def get_pfs_label(row):
#     months = row['PFS_month']
#     event = row['PFS_status']
    
#     if months >= 18:
#         return 2  # Long responder
#     elif months < 6 and event == 1:
#         return 0   # Non responder
#     elif 6 <= months < 18 and event == 1:
#         return 1 # Intermediate
#     else:
#         return np.nan   # Undetermind
    
# clinical['pfs_label'] = clinical.apply(get_pfs_label, axis=1)

# clinical = clinical.drop(['PFS_status', 'PFS_month'], axis=1)
# clinical

In [ ]:
# clinical.write_csv(f"../dataset/created/clinical_filtered/clinical.csv")